# Loading Modules

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
from pathlib import Path

* Το Ερευνητικό Περιεχόμενο ενδιαφέροντος αφορά την απάντηση στο εξής ερώτημα:
 - Σε ένα session (συνεδρία) / match, δεδομένων των data του παίκτη έως και το λεπτό x, ποια η πιθανότητα τραυματισμού έως το τέλος του session.

 - Με άλλα λόγια προσπαθώ να ορίσω το:  **P(INJURY | DATA up-to-minute X)**
   
* Για τις ανάγκες του εγχειρήματος, χρησιμοποιούμε το **Soccermon Dataset**: HTML('<a href="https://zenodo.org/records/10033832">Soccermon Dataset</a>')

  Το SoccerMon dataset έχει δημοσιευθεί σε peer-reviewed journal του Nature (“Scientific Data”) και είναι ελεύθερα προσβάσιμο με DOI και Zenodo link. Περιέχει ανθρώπινες αναφορές τραυματισμών (injury reports) σε συνδυασμό με GPS/biomechanical & load metrics, γεγονός που το καθιστά πλήρες dataset για injury prediction / injury risk modelling.


* To **Soccermon Dataset** περίεχει data παικτών 2 Ομάδων (encrypted) για δύο χρονιές (2020-2021) που αφορούν:
  - Game Performance (σε ημερήσιο level)
  - Ilness Data (σε ημερήσιο level)
  - Injury Data (σε ημερήσιο level)
  - Training Load Data (σε ημερήσιο level)
  - Wellness Data (σε ημερήσιο level)
  - Session Data (σε session level):
    Δεδομένα GPS / Heart Rate / Mechanical Load κατά την διάρκεια προπονήσεων, αγώνων αλλά και pre-match. Τα δεδομένα αυτά δίνονται ανά δευτερόλεπτο ή και λίγο παραπάνω....Για τις ανάγκες του εγχειρήματός μας, εμείς κάνουμε aggregation σε επίπεδο λεπτού. Για παράδειγμα, αν το Heart Rate ενός παίκτη δίνεται ανά δευτερόλεπτο, εμείς ορίζουμε τα εξής: **Heart Rate Mean** / **Heart Rate Std** / **Heart Rate Median** κλπ. Επαναλαμβάνουμε το ίδιο για όλες τις μεταβλητές

  - ΣΗΜΑΝΤΙΚΟ: Οι τραυματισμοί καταγράφονται σε ημερήσιο επίπεδο. Εμείς λοιπόν, για τις ανάγκες του εγχειρήματός μας κάνουμε την εξής θεώρηση: Αν μια μέρα στην οποία υπάρχει καταγεγραμμένος τραυματισμός, συμπίπτει με παιχνίδι / προπόνηση (συνεδρία ας το πούμε), τότε θεωρούμε ότι ο τραυματισμός οφείλεται σε αυτό το συγκεκριμένο session.

  - Για να είμαστε απολύτως ειλικρινής, αυτό δεν είναι 100% απαραίτητο...Ένας παίκτης μπορεί να είχε υποψία τραυματισμού και σε προηγούμενη συνεδρία και απλά να την κατέγραψε σε μεταγενέστερη ημερομηνία, θεωρούμε όμως ότι για να συμμετείχε στην επόμενη συνεδρία, ο τραυματισμός δεν ήταν ξεκάθαρα σοβαρός. Για να είμαστε σωστοί, θεωρούμε ότι: Καταγραφή Τραυματισμού την ίδια μέρα με ένα session αφορά το ίδιο το session.
 
  - Υπάρχουν μέρες που οι συνεδρίες είναι πολλές: Μια μέρα μπορεί να συνοδεύεται από 2 η και περισσότερες συνεδρίες. Επειδή δεν μπορούμε να ξέρουμε σε ποιο από τα πολλαπλά sessions έγινε ο τραυματισμός, κρατάμε μόνο τις μέρες εκείνες που συνοδεύονται από ένα session, ώστε να είμαστε σίγουροι ότι ο τραυματισμός που είχε καταγραφεί εκείνη την ημέρα αφορά το ίδιο το session. Ο τρόπος για να φιλτραρουμε με αυτόν τον τρόπο είναι να έχουμε ένα session που κρατά από 0 έως 
 

* Σκοπός της ανάλυσης αυτής είναι να δούμε αν υπάρχει **σήμα από τα δεδομένα**, ώστε να είμαστε σίγουροι πως περαιτέρω ανάλυση έχει βάση.
  Αν δούμε ότι οι μεταβλητές πράγματι συσχετίζονται με το injury, τότε η ανάλυση έχει νόημα.
  Αν όχι, τοτε δεν υπάρχει λόγος να προχωρήσουμε

### Defining Functions

In [2]:
def load_minute_file(path):
    df = pd.read_csv(path)
    df.columns = [c.strip() for c in df.columns]

    # rename FIRST
    df = df.rename(columns={
        'player_name_': 'player_name',
        'date_': 'date',
        'minute_': 'minute',
        'minute_idx_': 'minute_idx'
    })

    ID_COLS = ["player_name", "date", "minute", "minute_idx"]

    # fix id columns
    df["player_name"] = df["player_name"].astype("string")
    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    df["minute"] = pd.to_datetime(df["minute"], errors="coerce")
    df["minute_idx"] = pd.to_numeric(df["minute_idx"], errors="coerce").astype("Int64")

    # numeric features
    numeric_cols = [c for c in df.columns if c not in ID_COLS]
    df[numeric_cols] = df[numeric_cols].apply(pd.to_numeric, errors="coerce")

    return df


def add_session_id(df):
    """
    Adds:
      - team: first 5 chars of player_name (your convention)
      - session_id: team + "_" + date(YYYY-MM-DD ...)
    """
    return (
        df.assign(
            date=pd.to_datetime(df["date"], errors="coerce"),
            team=df["player_name"].astype(str).str[:5]
        )
        .assign(session_id=lambda x: x["team"] + "_" + x["date"].astype(str))
    )


def safe_left_merge(base, other, cols, name):
    other_small = other[cols].drop_duplicates()
    assert other_small.duplicated(["player_name", "session_id"]).sum() == 0, f"Duplicate keys in {name}"
    return base.merge(other_small, on=["player_name", "session_id"], how="left", validate="m:1")


def restrict_to_game_keys(df, keys):
    return df.merge(keys, on=["player_name", "session_id"], how="inner")


def lag_subjective(df, value_cols, lag_days=1):
    """
    Strict temporal precedence:
      - takes values recorded on day D-1
      - shifts date forward to day D
      - creates session_id for day D so that D-1 values merge onto session D.
    """
    out = df.copy()
    out["date"] = pd.to_datetime(out["date"], errors="coerce") + pd.to_timedelta(lag_days, unit="D")
    out = add_session_id(out)
    return out[["player_name", "session_id"] + value_cols]


def melt_daily(df, value_name):
    df = df.copy()

    # Βρες τη date column δυναμικά
    if "Date" in df.columns:
        date_col = "Date"
    elif "date" in df.columns:
        date_col = "date"
    else:
        raise ValueError(f"No date column found in dataframe for {value_name}")

    out = df.melt(
        id_vars=date_col,
        var_name="player_name",
        value_name=value_name
    )

    out.rename(columns={date_col: "date"}, inplace=True)
    return out

### Defining Main Paths

In [3]:
main_path = 'D:/PhD EMP'

# subjective
subjective_name = 'subjective'
game_performance = 'game-performance/game-performance.csv'
illness = 'illness/illness.csv'
injury  = 'injury/injury.csv'

acwr = 'training-load/acwr.csv'
atl = 'training-load/atl.csv'
ctl28 = 'training-load/ctl28.csv'
ctl42 = 'training-load/ctl42.csv'
daily_load = 'training-load/daily_load.csv'
monotony = 'training-load/monotony.csv'
strain = 'training-load/strain.csv'
weekly_load = 'training-load/weekly_load.csv'
session_path = 'training-load/session.json'

fatigue = 'wellness/fatigue.csv'
mood = 'wellness/mood.csv'
readiness = 'wellness/readiness.csv'
sleep_duration = 'wellness/sleep_duration.csv'
sleep_quality = 'wellness/sleep_quality.csv'
soreness = 'wellness/soreness.csv'
stress = 'wellness/stress.csv'


### Subjective Data Loading

In [4]:
game_performance_data = pd.read_csv(main_path + f'/{subjective_name}/{game_performance}')
illness_data = pd.read_csv(main_path + f'/{subjective_name}/{illness}')
injury_data = pd.read_csv(main_path + f'/{subjective_name}/{injury}')

acwr_data = pd.read_csv(main_path + f'/{subjective_name}/{acwr}')
atl_data = pd.read_csv(main_path + f'/{subjective_name}/{atl}')
ctl28_data = pd.read_csv(main_path + f'/{subjective_name}/{ctl28}')
ctl42_data = pd.read_csv(main_path + f'/{subjective_name}/{ctl42}')
daily_load_data = pd.read_csv(main_path + f'/{subjective_name}/{daily_load}')
monotony_data = pd.read_csv(main_path + f'/{subjective_name}/{monotony}')
strain_data = pd.read_csv(main_path + f'/{subjective_name}/{strain}')
weekly_load_data = pd.read_csv(main_path + f'/{subjective_name}/{weekly_load}')

fatigue_data = pd.read_csv(main_path + f'/{subjective_name}/{fatigue}')
mood_data = pd.read_csv(main_path + f'/{subjective_name}/{mood}')
readiness_data = pd.read_csv(main_path + f'/{subjective_name}/{readiness}')
sleep_duration_data = pd.read_csv(main_path + f'/{subjective_name}/{sleep_duration}')
sleep_quality_data = pd.read_csv(main_path + f'/{subjective_name}/{sleep_quality}')
soreness_data = pd.read_csv(main_path + f'/{subjective_name}/{soreness}')
stress_data = pd.read_csv(main_path + f'/{subjective_name}/{stress}')



with open(main_path + f'/{subjective_name}/{session_path}', "r") as f:
    session_dict = json.load(f)

rows = []
for player_name, sessions in session_dict.items():
    for s in sessions:
        rows.append({
            "player_name": player_name,
            "date": pd.to_datetime(s["date"], format="%d.%m.%Y"),
            "srpe": s["srpe"],
            "rpe": s["rpe"],
            "duration": s["duration"]
        })
session_data = pd.DataFrame(rows)

### Wellness Data Inspection

In [5]:
game_performance_data.head()

,player_name,team_performance,offensive_performance,defensive_performance,timestamp
0,TeamA-d7299614-fa73-4f69-b5e9-f913e3154ff6,7,5,6,11.07.2020
1,TeamA-d7299614-fa73-4f69-b5e9-f913e3154ff6,7,7,7,07.10.2020
2,TeamA-d7299614-fa73-4f69-b5e9-f913e3154ff6,5,7,5,18.10.2020
3,TeamA-d7299614-fa73-4f69-b5e9-f913e3154ff6,6,6,7,31.10.2020
4,TeamA-d7299614-fa73-4f69-b5e9-f913e3154ff6,4,4,4,07.11.2020


In [6]:
illness_data.head()

,player_name,problems,timestamp
0,TeamA-affd6f1d-b364-4700-98bf-8f20896e5ac4,"[""soar throat""]",16.09.2020
1,TeamA-3e5f6e2b-46b7-4890-84a9-3bbb2649af5a,"[""nausea""]",11.08.2020
2,TeamA-3e5f6e2b-46b7-4890-84a9-3bbb2649af5a,"[""headache"",""nausea"",""other""]",04.01.2021
3,TeamA-3e5f6e2b-46b7-4890-84a9-3bbb2649af5a,"[""soar throat""]",18.10.2021
4,TeamA-3e5f6e2b-46b7-4890-84a9-3bbb2649af5a,"[""headache"",""fever"",""soar throat"",""coughing""]",20.10.2021


In [7]:
injury_data["injury_date"] = pd.to_datetime(injury_data["timestamp"], dayfirst=True).dt.date
injury_data["player_name"] = injury_data["player_name"].astype("string")
injury_data.head()

,player_name,type,timestamp,injury_date
0,TeamA-b58af410-da77-479e-b93c-e03617b9f36d,"{""right_thigh"":""minor""}",20.03.2020,2020-03-20
1,TeamA-5cd7a61b-88b2-46d2-94f8-5a0d4f682d93,"{""right_foot"":""minor""}",20.03.2020,2020-03-20
2,TeamA-5cd7a61b-88b2-46d2-94f8-5a0d4f682d93,"{""right_foot"":""minor""}",21.03.2020,2020-03-21
3,TeamA-5cd7a61b-88b2-46d2-94f8-5a0d4f682d93,"{""right_foot"":""minor""}",21.03.2020,2020-03-21
4,TeamA-5cd7a61b-88b2-46d2-94f8-5a0d4f682d93,"{""right_foot"":""minor""}",22.03.2020,2020-03-22


In [8]:
acwr_data.head()

,Date,TeamA-d7299614-fa73-4f69-b5e9-f913e3154ff6,TeamA-b58af410-da77-479e-b93c-e03617b9f36d,TeamA-5cd7a61b-88b2-46d2-94f8-5a0d4f682d93,TeamA-74afe68c-f348-414c-9754-6d6f9df12587,TeamA-bcc03f81-2733-45d3-abf1-f7a709c63e68,TeamA-2d44f941-2f24-4fc2-afa8-611a091f2e93,TeamA-ecdbd8ec-61a7-4131-97ec-3af76f621f65,TeamA-e920ae60-5c4b-4597-be27-fc6876dcec33,TeamA-af719df9-3e6c-4ad4-9e8e-0c0c45f4f76a,...,TeamB-9c845827-99f9-47f4-977d-2200a107a013,TeamB-ab5ec6e5-b0f3-401e-8dc5-aa6c5070e255,TeamB-d9501f3a-aaa1-4311-8b8b-3a8d3f21480f,TeamB-5eba590a-cecb-400f-a92e-08224c7e2e59,TeamB-7d41e602-eece-328b-ff7b-118e820865d6,TeamB-4b1e23d2-4204-4e16-9991-e0490649079b,TeamB-e1f63352-01cb-4688-9486-5c6f1e5cf0cd,TeamB-5a921187-19c7-8df4-8f4f-f31e78de5857,TeamB-6d568bee-175f-4dcb-9d3a-0f3e8f35de07,TeamB-08e35b7b-d2ad-411f-a0de-4e663f14c3c3
0,01.01.2020,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,02.01.2020,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,03.01.2020,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,04.01.2020,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,05.01.2020,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [9]:
atl_data.head()

,Date,TeamA-d7299614-fa73-4f69-b5e9-f913e3154ff6,TeamA-b58af410-da77-479e-b93c-e03617b9f36d,TeamA-5cd7a61b-88b2-46d2-94f8-5a0d4f682d93,TeamA-74afe68c-f348-414c-9754-6d6f9df12587,TeamA-bcc03f81-2733-45d3-abf1-f7a709c63e68,TeamA-2d44f941-2f24-4fc2-afa8-611a091f2e93,TeamA-ecdbd8ec-61a7-4131-97ec-3af76f621f65,TeamA-e920ae60-5c4b-4597-be27-fc6876dcec33,TeamA-af719df9-3e6c-4ad4-9e8e-0c0c45f4f76a,...,TeamB-9c845827-99f9-47f4-977d-2200a107a013,TeamB-ab5ec6e5-b0f3-401e-8dc5-aa6c5070e255,TeamB-d9501f3a-aaa1-4311-8b8b-3a8d3f21480f,TeamB-5eba590a-cecb-400f-a92e-08224c7e2e59,TeamB-7d41e602-eece-328b-ff7b-118e820865d6,TeamB-4b1e23d2-4204-4e16-9991-e0490649079b,TeamB-e1f63352-01cb-4688-9486-5c6f1e5cf0cd,TeamB-5a921187-19c7-8df4-8f4f-f31e78de5857,TeamB-6d568bee-175f-4dcb-9d3a-0f3e8f35de07,TeamB-08e35b7b-d2ad-411f-a0de-4e663f14c3c3
0,01.01.2020,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,02.01.2020,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,03.01.2020,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,04.01.2020,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,05.01.2020,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [10]:
ctl28_data.head()

,Date,TeamA-d7299614-fa73-4f69-b5e9-f913e3154ff6,TeamA-b58af410-da77-479e-b93c-e03617b9f36d,TeamA-5cd7a61b-88b2-46d2-94f8-5a0d4f682d93,TeamA-74afe68c-f348-414c-9754-6d6f9df12587,TeamA-bcc03f81-2733-45d3-abf1-f7a709c63e68,TeamA-2d44f941-2f24-4fc2-afa8-611a091f2e93,TeamA-ecdbd8ec-61a7-4131-97ec-3af76f621f65,TeamA-e920ae60-5c4b-4597-be27-fc6876dcec33,TeamA-af719df9-3e6c-4ad4-9e8e-0c0c45f4f76a,...,TeamB-9c845827-99f9-47f4-977d-2200a107a013,TeamB-ab5ec6e5-b0f3-401e-8dc5-aa6c5070e255,TeamB-d9501f3a-aaa1-4311-8b8b-3a8d3f21480f,TeamB-5eba590a-cecb-400f-a92e-08224c7e2e59,TeamB-7d41e602-eece-328b-ff7b-118e820865d6,TeamB-4b1e23d2-4204-4e16-9991-e0490649079b,TeamB-e1f63352-01cb-4688-9486-5c6f1e5cf0cd,TeamB-5a921187-19c7-8df4-8f4f-f31e78de5857,TeamB-6d568bee-175f-4dcb-9d3a-0f3e8f35de07,TeamB-08e35b7b-d2ad-411f-a0de-4e663f14c3c3
0,01.01.2020,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,02.01.2020,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,03.01.2020,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,04.01.2020,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,05.01.2020,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [11]:
ctl42_data.head()

,Date,TeamA-d7299614-fa73-4f69-b5e9-f913e3154ff6,TeamA-b58af410-da77-479e-b93c-e03617b9f36d,TeamA-5cd7a61b-88b2-46d2-94f8-5a0d4f682d93,TeamA-74afe68c-f348-414c-9754-6d6f9df12587,TeamA-bcc03f81-2733-45d3-abf1-f7a709c63e68,TeamA-2d44f941-2f24-4fc2-afa8-611a091f2e93,TeamA-ecdbd8ec-61a7-4131-97ec-3af76f621f65,TeamA-e920ae60-5c4b-4597-be27-fc6876dcec33,TeamA-af719df9-3e6c-4ad4-9e8e-0c0c45f4f76a,...,TeamB-9c845827-99f9-47f4-977d-2200a107a013,TeamB-ab5ec6e5-b0f3-401e-8dc5-aa6c5070e255,TeamB-d9501f3a-aaa1-4311-8b8b-3a8d3f21480f,TeamB-5eba590a-cecb-400f-a92e-08224c7e2e59,TeamB-7d41e602-eece-328b-ff7b-118e820865d6,TeamB-4b1e23d2-4204-4e16-9991-e0490649079b,TeamB-e1f63352-01cb-4688-9486-5c6f1e5cf0cd,TeamB-5a921187-19c7-8df4-8f4f-f31e78de5857,TeamB-6d568bee-175f-4dcb-9d3a-0f3e8f35de07,TeamB-08e35b7b-d2ad-411f-a0de-4e663f14c3c3
0,01.01.2020,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,02.01.2020,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,03.01.2020,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,04.01.2020,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,05.01.2020,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [12]:
daily_load_data.head()

,Date,TeamA-d7299614-fa73-4f69-b5e9-f913e3154ff6,TeamA-b58af410-da77-479e-b93c-e03617b9f36d,TeamA-5cd7a61b-88b2-46d2-94f8-5a0d4f682d93,TeamA-74afe68c-f348-414c-9754-6d6f9df12587,TeamA-bcc03f81-2733-45d3-abf1-f7a709c63e68,TeamA-2d44f941-2f24-4fc2-afa8-611a091f2e93,TeamA-ecdbd8ec-61a7-4131-97ec-3af76f621f65,TeamA-e920ae60-5c4b-4597-be27-fc6876dcec33,TeamA-af719df9-3e6c-4ad4-9e8e-0c0c45f4f76a,...,TeamB-9c845827-99f9-47f4-977d-2200a107a013,TeamB-ab5ec6e5-b0f3-401e-8dc5-aa6c5070e255,TeamB-d9501f3a-aaa1-4311-8b8b-3a8d3f21480f,TeamB-5eba590a-cecb-400f-a92e-08224c7e2e59,TeamB-7d41e602-eece-328b-ff7b-118e820865d6,TeamB-4b1e23d2-4204-4e16-9991-e0490649079b,TeamB-e1f63352-01cb-4688-9486-5c6f1e5cf0cd,TeamB-5a921187-19c7-8df4-8f4f-f31e78de5857,TeamB-6d568bee-175f-4dcb-9d3a-0f3e8f35de07,TeamB-08e35b7b-d2ad-411f-a0de-4e663f14c3c3
0,01.01.2020,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,02.01.2020,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,03.01.2020,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,04.01.2020,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,05.01.2020,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [13]:
monotony_data.head()

,Date,TeamA-d7299614-fa73-4f69-b5e9-f913e3154ff6,TeamA-b58af410-da77-479e-b93c-e03617b9f36d,TeamA-5cd7a61b-88b2-46d2-94f8-5a0d4f682d93,TeamA-74afe68c-f348-414c-9754-6d6f9df12587,TeamA-bcc03f81-2733-45d3-abf1-f7a709c63e68,TeamA-2d44f941-2f24-4fc2-afa8-611a091f2e93,TeamA-ecdbd8ec-61a7-4131-97ec-3af76f621f65,TeamA-e920ae60-5c4b-4597-be27-fc6876dcec33,TeamA-af719df9-3e6c-4ad4-9e8e-0c0c45f4f76a,...,TeamB-9c845827-99f9-47f4-977d-2200a107a013,TeamB-ab5ec6e5-b0f3-401e-8dc5-aa6c5070e255,TeamB-d9501f3a-aaa1-4311-8b8b-3a8d3f21480f,TeamB-5eba590a-cecb-400f-a92e-08224c7e2e59,TeamB-7d41e602-eece-328b-ff7b-118e820865d6,TeamB-4b1e23d2-4204-4e16-9991-e0490649079b,TeamB-e1f63352-01cb-4688-9486-5c6f1e5cf0cd,TeamB-5a921187-19c7-8df4-8f4f-f31e78de5857,TeamB-6d568bee-175f-4dcb-9d3a-0f3e8f35de07,TeamB-08e35b7b-d2ad-411f-a0de-4e663f14c3c3
0,01.01.2020,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,02.01.2020,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,03.01.2020,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,04.01.2020,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,05.01.2020,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [14]:
strain_data.head()

,Date,TeamA-d7299614-fa73-4f69-b5e9-f913e3154ff6,TeamA-b58af410-da77-479e-b93c-e03617b9f36d,TeamA-5cd7a61b-88b2-46d2-94f8-5a0d4f682d93,TeamA-74afe68c-f348-414c-9754-6d6f9df12587,TeamA-bcc03f81-2733-45d3-abf1-f7a709c63e68,TeamA-2d44f941-2f24-4fc2-afa8-611a091f2e93,TeamA-ecdbd8ec-61a7-4131-97ec-3af76f621f65,TeamA-e920ae60-5c4b-4597-be27-fc6876dcec33,TeamA-af719df9-3e6c-4ad4-9e8e-0c0c45f4f76a,...,TeamB-9c845827-99f9-47f4-977d-2200a107a013,TeamB-ab5ec6e5-b0f3-401e-8dc5-aa6c5070e255,TeamB-d9501f3a-aaa1-4311-8b8b-3a8d3f21480f,TeamB-5eba590a-cecb-400f-a92e-08224c7e2e59,TeamB-7d41e602-eece-328b-ff7b-118e820865d6,TeamB-4b1e23d2-4204-4e16-9991-e0490649079b,TeamB-e1f63352-01cb-4688-9486-5c6f1e5cf0cd,TeamB-5a921187-19c7-8df4-8f4f-f31e78de5857,TeamB-6d568bee-175f-4dcb-9d3a-0f3e8f35de07,TeamB-08e35b7b-d2ad-411f-a0de-4e663f14c3c3
0,01.01.2020,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,02.01.2020,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,03.01.2020,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,04.01.2020,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,05.01.2020,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [15]:
weekly_load_data.head()

,Date,TeamA-d7299614-fa73-4f69-b5e9-f913e3154ff6,TeamA-b58af410-da77-479e-b93c-e03617b9f36d,TeamA-5cd7a61b-88b2-46d2-94f8-5a0d4f682d93,TeamA-74afe68c-f348-414c-9754-6d6f9df12587,TeamA-bcc03f81-2733-45d3-abf1-f7a709c63e68,TeamA-2d44f941-2f24-4fc2-afa8-611a091f2e93,TeamA-ecdbd8ec-61a7-4131-97ec-3af76f621f65,TeamA-e920ae60-5c4b-4597-be27-fc6876dcec33,TeamA-af719df9-3e6c-4ad4-9e8e-0c0c45f4f76a,...,TeamB-9c845827-99f9-47f4-977d-2200a107a013,TeamB-ab5ec6e5-b0f3-401e-8dc5-aa6c5070e255,TeamB-d9501f3a-aaa1-4311-8b8b-3a8d3f21480f,TeamB-5eba590a-cecb-400f-a92e-08224c7e2e59,TeamB-7d41e602-eece-328b-ff7b-118e820865d6,TeamB-4b1e23d2-4204-4e16-9991-e0490649079b,TeamB-e1f63352-01cb-4688-9486-5c6f1e5cf0cd,TeamB-5a921187-19c7-8df4-8f4f-f31e78de5857,TeamB-6d568bee-175f-4dcb-9d3a-0f3e8f35de07,TeamB-08e35b7b-d2ad-411f-a0de-4e663f14c3c3
0,01.01.2020,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,02.01.2020,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,03.01.2020,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,04.01.2020,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,05.01.2020,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [16]:
fatigue_data.rename(columns={"Fatigue Data": "Date"}, inplace=True)
fatigue_data.head()

,Date,TeamA-d7299614-fa73-4f69-b5e9-f913e3154ff6,TeamA-b58af410-da77-479e-b93c-e03617b9f36d,TeamA-5cd7a61b-88b2-46d2-94f8-5a0d4f682d93,TeamA-74afe68c-f348-414c-9754-6d6f9df12587,TeamA-bcc03f81-2733-45d3-abf1-f7a709c63e68,TeamA-2d44f941-2f24-4fc2-afa8-611a091f2e93,TeamA-ecdbd8ec-61a7-4131-97ec-3af76f621f65,TeamA-e920ae60-5c4b-4597-be27-fc6876dcec33,TeamA-af719df9-3e6c-4ad4-9e8e-0c0c45f4f76a,...,TeamB-9c845827-99f9-47f4-977d-2200a107a013,TeamB-ab5ec6e5-b0f3-401e-8dc5-aa6c5070e255,TeamB-d9501f3a-aaa1-4311-8b8b-3a8d3f21480f,TeamB-5eba590a-cecb-400f-a92e-08224c7e2e59,TeamB-7d41e602-eece-328b-ff7b-118e820865d6,TeamB-4b1e23d2-4204-4e16-9991-e0490649079b,TeamB-e1f63352-01cb-4688-9486-5c6f1e5cf0cd,TeamB-5a921187-19c7-8df4-8f4f-f31e78de5857,TeamB-6d568bee-175f-4dcb-9d3a-0f3e8f35de07,TeamB-08e35b7b-d2ad-411f-a0de-4e663f14c3c3
0,01.01.2020,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,02.01.2020,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,03.01.2020,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,04.01.2020,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,05.01.2020,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [17]:
mood_data.rename(columns={"Mood Data": "Date"}, inplace=True)
mood_data.head()

,Date,TeamA-d7299614-fa73-4f69-b5e9-f913e3154ff6,TeamA-b58af410-da77-479e-b93c-e03617b9f36d,TeamA-5cd7a61b-88b2-46d2-94f8-5a0d4f682d93,TeamA-74afe68c-f348-414c-9754-6d6f9df12587,TeamA-bcc03f81-2733-45d3-abf1-f7a709c63e68,TeamA-2d44f941-2f24-4fc2-afa8-611a091f2e93,TeamA-ecdbd8ec-61a7-4131-97ec-3af76f621f65,TeamA-e920ae60-5c4b-4597-be27-fc6876dcec33,TeamA-af719df9-3e6c-4ad4-9e8e-0c0c45f4f76a,...,TeamB-9c845827-99f9-47f4-977d-2200a107a013,TeamB-ab5ec6e5-b0f3-401e-8dc5-aa6c5070e255,TeamB-d9501f3a-aaa1-4311-8b8b-3a8d3f21480f,TeamB-5eba590a-cecb-400f-a92e-08224c7e2e59,TeamB-7d41e602-eece-328b-ff7b-118e820865d6,TeamB-4b1e23d2-4204-4e16-9991-e0490649079b,TeamB-e1f63352-01cb-4688-9486-5c6f1e5cf0cd,TeamB-5a921187-19c7-8df4-8f4f-f31e78de5857,TeamB-6d568bee-175f-4dcb-9d3a-0f3e8f35de07,TeamB-08e35b7b-d2ad-411f-a0de-4e663f14c3c3
0,01.01.2020,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,02.01.2020,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,03.01.2020,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,04.01.2020,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,05.01.2020,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [18]:
readiness_data.rename(columns={"Readiness Data": "Date"}, inplace=True)
readiness_data.head()

,Date,TeamA-d7299614-fa73-4f69-b5e9-f913e3154ff6,TeamA-b58af410-da77-479e-b93c-e03617b9f36d,TeamA-5cd7a61b-88b2-46d2-94f8-5a0d4f682d93,TeamA-74afe68c-f348-414c-9754-6d6f9df12587,TeamA-bcc03f81-2733-45d3-abf1-f7a709c63e68,TeamA-2d44f941-2f24-4fc2-afa8-611a091f2e93,TeamA-ecdbd8ec-61a7-4131-97ec-3af76f621f65,TeamA-e920ae60-5c4b-4597-be27-fc6876dcec33,TeamA-af719df9-3e6c-4ad4-9e8e-0c0c45f4f76a,...,TeamB-9c845827-99f9-47f4-977d-2200a107a013,TeamB-ab5ec6e5-b0f3-401e-8dc5-aa6c5070e255,TeamB-d9501f3a-aaa1-4311-8b8b-3a8d3f21480f,TeamB-5eba590a-cecb-400f-a92e-08224c7e2e59,TeamB-7d41e602-eece-328b-ff7b-118e820865d6,TeamB-4b1e23d2-4204-4e16-9991-e0490649079b,TeamB-e1f63352-01cb-4688-9486-5c6f1e5cf0cd,TeamB-5a921187-19c7-8df4-8f4f-f31e78de5857,TeamB-6d568bee-175f-4dcb-9d3a-0f3e8f35de07,TeamB-08e35b7b-d2ad-411f-a0de-4e663f14c3c3
0,01.01.2020,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,02.01.2020,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,03.01.2020,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,04.01.2020,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,05.01.2020,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [19]:
sleep_duration_data.rename(columns={"SleepDurH Data": "Date"}, inplace=True)
sleep_duration_data.head()

,Date,TeamA-d7299614-fa73-4f69-b5e9-f913e3154ff6,TeamA-b58af410-da77-479e-b93c-e03617b9f36d,TeamA-5cd7a61b-88b2-46d2-94f8-5a0d4f682d93,TeamA-74afe68c-f348-414c-9754-6d6f9df12587,TeamA-bcc03f81-2733-45d3-abf1-f7a709c63e68,TeamA-2d44f941-2f24-4fc2-afa8-611a091f2e93,TeamA-ecdbd8ec-61a7-4131-97ec-3af76f621f65,TeamA-e920ae60-5c4b-4597-be27-fc6876dcec33,TeamA-af719df9-3e6c-4ad4-9e8e-0c0c45f4f76a,...,TeamB-9c845827-99f9-47f4-977d-2200a107a013,TeamB-ab5ec6e5-b0f3-401e-8dc5-aa6c5070e255,TeamB-d9501f3a-aaa1-4311-8b8b-3a8d3f21480f,TeamB-5eba590a-cecb-400f-a92e-08224c7e2e59,TeamB-7d41e602-eece-328b-ff7b-118e820865d6,TeamB-4b1e23d2-4204-4e16-9991-e0490649079b,TeamB-e1f63352-01cb-4688-9486-5c6f1e5cf0cd,TeamB-5a921187-19c7-8df4-8f4f-f31e78de5857,TeamB-6d568bee-175f-4dcb-9d3a-0f3e8f35de07,TeamB-08e35b7b-d2ad-411f-a0de-4e663f14c3c3
0,01.01.2020,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,02.01.2020,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,03.01.2020,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,04.01.2020,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,05.01.2020,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [20]:
sleep_quality_data.rename(columns={"SleepQuality Data": "Date"}, inplace=True)
sleep_quality_data.head()

,Date,TeamA-d7299614-fa73-4f69-b5e9-f913e3154ff6,TeamA-b58af410-da77-479e-b93c-e03617b9f36d,TeamA-5cd7a61b-88b2-46d2-94f8-5a0d4f682d93,TeamA-74afe68c-f348-414c-9754-6d6f9df12587,TeamA-bcc03f81-2733-45d3-abf1-f7a709c63e68,TeamA-2d44f941-2f24-4fc2-afa8-611a091f2e93,TeamA-ecdbd8ec-61a7-4131-97ec-3af76f621f65,TeamA-e920ae60-5c4b-4597-be27-fc6876dcec33,TeamA-af719df9-3e6c-4ad4-9e8e-0c0c45f4f76a,...,TeamB-9c845827-99f9-47f4-977d-2200a107a013,TeamB-ab5ec6e5-b0f3-401e-8dc5-aa6c5070e255,TeamB-d9501f3a-aaa1-4311-8b8b-3a8d3f21480f,TeamB-5eba590a-cecb-400f-a92e-08224c7e2e59,TeamB-7d41e602-eece-328b-ff7b-118e820865d6,TeamB-4b1e23d2-4204-4e16-9991-e0490649079b,TeamB-e1f63352-01cb-4688-9486-5c6f1e5cf0cd,TeamB-5a921187-19c7-8df4-8f4f-f31e78de5857,TeamB-6d568bee-175f-4dcb-9d3a-0f3e8f35de07,TeamB-08e35b7b-d2ad-411f-a0de-4e663f14c3c3
0,01.01.2020,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,02.01.2020,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,03.01.2020,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,04.01.2020,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,05.01.2020,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [21]:
stress_data.head()

,Date,TeamA-d7299614-fa73-4f69-b5e9-f913e3154ff6,TeamA-b58af410-da77-479e-b93c-e03617b9f36d,TeamA-5cd7a61b-88b2-46d2-94f8-5a0d4f682d93,TeamA-74afe68c-f348-414c-9754-6d6f9df12587,TeamA-bcc03f81-2733-45d3-abf1-f7a709c63e68,TeamA-2d44f941-2f24-4fc2-afa8-611a091f2e93,TeamA-ecdbd8ec-61a7-4131-97ec-3af76f621f65,TeamA-e920ae60-5c4b-4597-be27-fc6876dcec33,TeamA-af719df9-3e6c-4ad4-9e8e-0c0c45f4f76a,...,TeamB-9c845827-99f9-47f4-977d-2200a107a013,TeamB-ab5ec6e5-b0f3-401e-8dc5-aa6c5070e255,TeamB-d9501f3a-aaa1-4311-8b8b-3a8d3f21480f,TeamB-5eba590a-cecb-400f-a92e-08224c7e2e59,TeamB-7d41e602-eece-328b-ff7b-118e820865d6,TeamB-4b1e23d2-4204-4e16-9991-e0490649079b,TeamB-e1f63352-01cb-4688-9486-5c6f1e5cf0cd,TeamB-5a921187-19c7-8df4-8f4f-f31e78de5857,TeamB-6d568bee-175f-4dcb-9d3a-0f3e8f35de07,TeamB-08e35b7b-d2ad-411f-a0de-4e663f14c3c3
0,01.01.2020,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,02.01.2020,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,03.01.2020,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,04.01.2020,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,05.01.2020,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [22]:
soreness_data.rename(columns={"Soreness Data": "Date"}, inplace=True)
soreness_data.head()

,Date,TeamA-d7299614-fa73-4f69-b5e9-f913e3154ff6,TeamA-b58af410-da77-479e-b93c-e03617b9f36d,TeamA-5cd7a61b-88b2-46d2-94f8-5a0d4f682d93,TeamA-74afe68c-f348-414c-9754-6d6f9df12587,TeamA-bcc03f81-2733-45d3-abf1-f7a709c63e68,TeamA-2d44f941-2f24-4fc2-afa8-611a091f2e93,TeamA-ecdbd8ec-61a7-4131-97ec-3af76f621f65,TeamA-e920ae60-5c4b-4597-be27-fc6876dcec33,TeamA-af719df9-3e6c-4ad4-9e8e-0c0c45f4f76a,...,TeamB-9c845827-99f9-47f4-977d-2200a107a013,TeamB-ab5ec6e5-b0f3-401e-8dc5-aa6c5070e255,TeamB-d9501f3a-aaa1-4311-8b8b-3a8d3f21480f,TeamB-5eba590a-cecb-400f-a92e-08224c7e2e59,TeamB-7d41e602-eece-328b-ff7b-118e820865d6,TeamB-4b1e23d2-4204-4e16-9991-e0490649079b,TeamB-e1f63352-01cb-4688-9486-5c6f1e5cf0cd,TeamB-5a921187-19c7-8df4-8f4f-f31e78de5857,TeamB-6d568bee-175f-4dcb-9d3a-0f3e8f35de07,TeamB-08e35b7b-d2ad-411f-a0de-4e663f14c3c3
0,01.01.2020,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,02.01.2020,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,03.01.2020,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,04.01.2020,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,05.01.2020,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Wide Datasets to Long Format

In [23]:
acwr_data       = melt_daily(acwr_data, "acwr")
atl_data        = melt_daily(atl_data, "atl")
ctl28_data      = melt_daily(ctl28_data, "ctl28")
ctl42_data      = melt_daily(ctl42_data, "ctl42")
daily_load_data  = melt_daily(daily_load_data, "daily_load")
weekly_load_data = melt_daily(weekly_load_data, "weekly_load")
monotony_data    = melt_daily(monotony_data, "monotony")
strain_data      = melt_daily(strain_data, "strain")

fatigue_data        = melt_daily(fatigue_data, "fatigue")
mood_data           = melt_daily(mood_data, "mood")
readiness_data      = melt_daily(readiness_data, "readiness")
sleep_duration_data = melt_daily(sleep_duration_data, "sleep_duration")
sleep_quality_data  = melt_daily(sleep_quality_data, "sleep_quality")
soreness_data       = melt_daily(soreness_data, "soreness")
stress_data         = melt_daily(stress_data, "stress")


game_performance_data.rename(columns={'timestamp': 'date'}, inplace=True)
illness_data.rename(columns={'timestamp': 'date'}, inplace=True)
injury_data.rename(columns={'timestamp': 'date'}, inplace=True)

### Load Minute Level Aggregates

In [24]:
root_dir = Path("D:/PhD EMP/2020")
minute_files = sorted(root_dir.rglob("minute_aggregates.csv"))

print("Minute files found:", len(minute_files))
if len(minute_files) > 0:
    print("Example file:", minute_files[0])


Minute files found: 167
Example file: D:\PhD EMP\2020\2020-06\2020-06-01\minute_aggregates.csv


### Preprocessing Minute Level Aggregates

In [25]:
dfs = [load_minute_file(f) for f in minute_files]
minute_2020_data = pd.concat(dfs, ignore_index=True)

# basic sanity
minute_2020_data = minute_2020_data.copy()
minute_2020_data["date"] = pd.to_datetime(minute_2020_data["date"]).dt.date
minute_2020_data["minute"] = pd.to_datetime(minute_2020_data["minute"], errors="coerce")

minute_2020_data = minute_2020_data[
    minute_2020_data["player_name"].notna() &
    minute_2020_data["date"].notna() &
    minute_2020_data["minute"].notna()
].reset_index(drop=True)

minute_2020_data["team"] = (
    minute_2020_data["player_name"].astype(str)
    .str.split("-", n=1)
    .str[0]
)

# =========================
# DEFINE EXPOSURE-BASED SESSION (TEAM × DATE)
# =========================

minute_2020_data["session_id"] = (
    minute_2020_data["team"].astype(str)
    + "_"
    + minute_2020_data["date"].astype(str)
)

# =========================
# COMPUTE DAILY EXPOSURE DURATION
# =========================

session_duration = (
    minute_2020_data
    .groupby(["team", "date"])
    .agg(
        start_time=("minute", "min"),
        end_time=("minute", "max"),
    )
    .reset_index()
)

session_duration["duration_minutes"] = (
    session_duration["end_time"] - session_duration["start_time"]
).dt.total_seconds() / 60

# =========================
# KEEP VALID EXPOSURE WINDOWS
# =========================

# NOTE: you decided to use 70..500 based on your exploration
session_days = session_duration.query(
    "duration_minutes >= 70 and duration_minutes <= 240"
)[["team", "date"]]

minute_2020_data = (
    minute_2020_data
    .merge(session_days, on=["team", "date"], how="inner")
    .reset_index(drop=True)
).drop_duplicates(['player_name','date','minute_idx'])

C:\Users\User\AppData\Local\Temp\ipykernel_2696\2973528084.py:2: DtypeWarning: Columns (3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path)


* Session Definition.
* Sessions were defined as contiguous exposure windows at the team–day level, representing continuous periods of physical load. Specifically, all minute-level data within the same team and calendar day were treated as a single exposure window. Only exposure windows lasting between 70 and 180 minutes were retained, corresponding to match-like or high-load training days. Multiple training bouts occurring within the same day were therefore considered as a single continuous exposure window, reflecting the cumulative nature of mechanical and physiological load. This exposure-based session definition enables minute-level modeling of injury risk while avoiding ambiguity in injury attribution across multiple within-day sessions.

In [26]:
minute_2020_data.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 352287 entries, 0 to 388586
Data columns (total 66 columns):
 #   Column                 Non-Null Count   Dtype         
---  ------                 --------------   -----         
 0   player_name            352287 non-null  string        
 1   date                   352287 non-null  object        
 2   minute                 352287 non-null  datetime64[ns]
 3   minute_idx             352287 non-null  Int64         
 4   lat_mean               352287 non-null  float64       
 5   lat_std                352287 non-null  float64       
 6   lat_min                352287 non-null  float64       
 7   lat_max                352287 non-null  float64       
 8   lon_mean               352287 non-null  float64       
 9   lon_std                352287 non-null  float64       
 10  lon_min                352287 non-null  float64       
 11  lon_max                352287 non-null  float64       
 12  speed_mean             352287 non-null  floa

### Aggregating on a Day-Session Level

In [27]:
minute_session = (
    minute_2020_data
    .groupby(["player_name", "team", "session_id"], as_index=False)
    .agg(
        # exposure
        total_minutes=("minute_idx", "max"),

        # =================
        # SPEED
        # =================
        speed_mean_sess=("speed_mean", "mean"),
        speed_sum_sess=("speed_mean", "sum"),

        # =================
        # HEART RATE
        # =================
        hr_mean_sess=("heart_rate_mean", "mean"),
        hr_sum_sess=("heart_rate_mean", "sum"),

        # =================
        # INSTANT ACC IMPULSE
        # =================
        inst_acc_mean_sess=("inst_acc_impulse_mean", "mean"),
        inst_acc_sum_sess=("inst_acc_impulse_mean", "sum"),

        # =================
        # HACC
        # =================
        hacc_mean_sess=("hacc_mean", "mean"),
        hacc_sum_sess=("hacc_mean", "sum"),

        # =================
        # ACCELEROMETER AXES
        # =================
        accl_x_mean_sess=("accl_x_mean", "mean"),
        accl_x_sum_sess=("accl_x_mean", "sum"),

        accl_y_mean_sess=("accl_y_mean", "mean"),
        accl_y_sum_sess=("accl_y_mean", "sum"),

        accl_z_mean_sess=("accl_z_mean", "mean"),
        accl_z_sum_sess=("accl_z_mean", "sum"),

        # =================
        # GYROSCOPE AXES
        # =================
        gyro_x_mean_sess=("gyro_x_mean", "mean"),
        gyro_x_sum_sess=("gyro_x_mean", "sum"),

        gyro_y_mean_sess=("gyro_y_mean", "mean"),
        gyro_y_sum_sess=("gyro_y_mean", "sum"),

        gyro_z_mean_sess=("gyro_z_mean", "mean"),
        gyro_z_sum_sess=("gyro_z_mean", "sum"),
    )
)

session_daily = (
    session_data
    .assign(date=pd.to_datetime(session_data["date"], errors="coerce"))
    .groupby(["player_name", "date"], as_index=False)
    .agg(
        srpe_sum=("srpe", "sum"),
        duration_sum=("duration", "sum"),
        rpe_mean=("rpe", "mean"),
        rpe_max=("rpe", "max"),
        n_sessions=("rpe", "count"),
    )
    .pipe(add_session_id)
)
acwr_data        = add_session_id(acwr_data)
atl_data         = add_session_id(atl_data)
ctl28_data       = add_session_id(ctl28_data)
ctl42_data       = add_session_id(ctl42_data)
daily_load_data  = add_session_id(daily_load_data)
monotony_data    = add_session_id(monotony_data)
strain_data      = add_session_id(strain_data)
weekly_load_data = add_session_id(weekly_load_data)

# sleep = SAME-DAY (previous night → pre-session)
sleep_duration_data = add_session_id(sleep_duration_data)
sleep_quality_data  = add_session_id(sleep_quality_data)

game_performance_data["date"] = pd.to_datetime(
    game_performance_data["date"], dayfirst=True, errors="coerce"
)
game_performance_data = add_session_id(game_performance_data)

injury_daily = (
    injury_data
    .assign(date=pd.to_datetime(injury_data["date"], errors="coerce"),
            injury=1)
    .groupby(["player_name", "date"], as_index=False)["injury"]
    .max()
    .pipe(add_session_id)
)

illness_daily = (
    illness_data
    .assign(date=pd.to_datetime(illness_data["date"], errors="coerce"),
            illness=1)
    .groupby(["player_name", "date"], as_index=False)["illness"]
    .max()
    .pipe(add_session_id)
)


fatigue_lag   = lag_subjective(
    fatigue_data[["player_name","date","fatigue"]],
    ["fatigue"], lag_days=1
)

mood_lag      = lag_subjective(
    mood_data[["player_name","date","mood"]],
    ["mood"], lag_days=1
)

readiness_lag = lag_subjective(
    readiness_data[["player_name","date","readiness"]],
    ["readiness"], lag_days=1
)

soreness_lag  = lag_subjective(
    soreness_data[["player_name","date","soreness"]],
    ["soreness"], lag_days=1
)

stress_lag    = lag_subjective(
    stress_data[["player_name","date","stress"]],
    ["stress"], 1
)

C:\Users\User\AppData\Local\Temp\ipykernel_2696\2973528084.py:36: UserWarning: Parsing dates in DD/MM/YYYY format when dayfirst=False (the default) was specified. This may lead to inconsistently parsed dates! Specify a format to ensure consistent parsing.
  date=pd.to_datetime(df["date"], errors="coerce"),
C:\Users\User\AppData\Local\Temp\ipykernel_2696\2973528084.py:36: UserWarning: Parsing dates in DD/MM/YYYY format when dayfirst=False (the default) was specified. This may lead to inconsistently parsed dates! Specify a format to ensure consistent parsing.
  date=pd.to_datetime(df["date"], errors="coerce"),
C:\Users\User\AppData\Local\Temp\ipykernel_2696\2973528084.py:36: UserWarning: Parsing dates in DD/MM/YYYY format when dayfirst=False (the default) was specified. This may lead to inconsistently parsed dates! Specify a format to ensure consistent parsing.
  date=pd.to_datetime(df["date"], errors="coerce"),
C:\Users\User\AppData\Local\Temp\ipykernel_2696\2973528084.py:36: UserWarnin

### Restricting to Game Sessions

In [28]:
# Keys που ορίζουν το dataset (μόνο game sessions)
keys = minute_session[["player_name", "session_id"]].drop_duplicates()
session_daily_g = restrict_to_game_keys(session_daily, keys)

atl_g       = restrict_to_game_keys(atl_data[["player_name","session_id","atl"]], keys)
ctl28_g     = restrict_to_game_keys(ctl28_data[["player_name","session_id","ctl28"]], keys)
ctl42_g     = restrict_to_game_keys(ctl42_data[["player_name","session_id","ctl42"]], keys)
daily_load_g = restrict_to_game_keys(daily_load_data[["player_name","session_id","daily_load"]], keys)
monotony_g  = restrict_to_game_keys(monotony_data[["player_name","session_id","monotony"]], keys)
strain_g    = restrict_to_game_keys(strain_data[["player_name","session_id","strain"]], keys)
weekly_load_g = restrict_to_game_keys(weekly_load_data[["player_name","session_id","weekly_load"]], keys)
acwr_g      = restrict_to_game_keys(acwr_data[["player_name","session_id","acwr"]], keys)

# Sleep same-day
sleep_dur_g = restrict_to_game_keys(sleep_duration_data[["player_name","session_id","sleep_duration"]], keys)
sleep_q_g   = restrict_to_game_keys(sleep_quality_data[["player_name","session_id","sleep_quality"]], keys)

# Lagged wellness
fatigue_g   = restrict_to_game_keys(fatigue_lag, keys)
mood_g      = restrict_to_game_keys(mood_lag, keys)
readiness_g = restrict_to_game_keys(readiness_lag, keys)
soreness_g  = restrict_to_game_keys(soreness_lag, keys)
stress_g    = restrict_to_game_keys(stress_lag, keys)

# Labels
injury_g  = restrict_to_game_keys(injury_daily[["player_name","session_id","injury"]], keys)
illness_g = restrict_to_game_keys(illness_daily[["player_name","session_id","illness"]], keys)

### Defining Master Level Day Session: 

In [29]:
master_session = safe_left_merge(
    minute_session, session_daily_g,
    ["player_name","session_id","srpe_sum","duration_sum","rpe_mean","rpe_max","n_sessions"],
    "session_daily"
)

master_session = safe_left_merge(master_session, atl_g,        ["player_name","session_id","atl"], "atl")
master_session = safe_left_merge(master_session, ctl28_g,      ["player_name","session_id","ctl28"], "ctl28")
master_session = safe_left_merge(master_session, ctl42_g,      ["player_name","session_id","ctl42"], "ctl42")
master_session = safe_left_merge(master_session, daily_load_g, ["player_name","session_id","daily_load"], "daily_load")
master_session = safe_left_merge(master_session, monotony_g,   ["player_name","session_id","monotony"], "monotony")
master_session = safe_left_merge(master_session, strain_g,     ["player_name","session_id","strain"], "strain")
master_session = safe_left_merge(master_session, weekly_load_g,["player_name","session_id","weekly_load"], "weekly_load")
master_session = safe_left_merge(master_session, acwr_g,       ["player_name","session_id","acwr"], "acwr")

# sleep same-day
master_session = safe_left_merge(master_session, sleep_dur_g, ["player_name","session_id","sleep_duration"], "sleep_duration")
master_session = safe_left_merge(master_session, sleep_q_g,   ["player_name","session_id","sleep_quality"], "sleep_quality")

# wellness lagged
master_session = safe_left_merge(master_session, fatigue_g,   ["player_name","session_id","fatigue"], "fatigue_lag1")
master_session = safe_left_merge(master_session, mood_g,      ["player_name","session_id","mood"], "mood_lag1")
master_session = safe_left_merge(master_session, readiness_g, ["player_name","session_id","readiness"], "readiness_lag1")
master_session = safe_left_merge(master_session, soreness_g,  ["player_name","session_id","soreness"], "soreness_lag1")
master_session = safe_left_merge(master_session, stress_g,    ["player_name","session_id","stress"], "stress_lag1")

# labels
master_session = safe_left_merge(master_session, injury_g,  ["player_name","session_id","injury"], "injury")
master_session = safe_left_merge(master_session, illness_g, ["player_name","session_id","illness"], "illness")

master_session["injury"] = master_session["injury"].fillna(0).astype(int)
master_session["illness"] = master_session["illness"].fillna(0).astype(int)

### Assertions

In [30]:
assert master_session.duplicated(["player_name","session_id"]).sum() == 0
print("Rows in master_session:", len(master_session))
print("Injuries in master_session:", int(master_session["injury"].sum()))
print(master_session["injury"].value_counts(normalize=True))

# Missingness quick view (optional)
print("\nTop missingness:")
print(master_session.isna().mean().sort_values(ascending=False).head(15))

Rows in master_session: 3515
Injuries in master_session: 24
0    0.993172
1    0.006828
Name: injury, dtype: float64

Top missingness:
sleep_quality     0.512945
sleep_duration    0.512945
fatigue           0.512660
readiness         0.512376
stress            0.512091
soreness          0.512091
mood              0.512091
srpe_sum          0.398293
duration_sum      0.398293
rpe_mean          0.398293
rpe_max           0.398293
n_sessions        0.398293
ctl42             0.304694
atl               0.304694
ctl28             0.304694
dtype: float64


### Writting Master Session in Folder

In [31]:
master_session.to_csv('D:/PhD EMP/2020/master_session.csv')
minute_2020_data.to_csv('D:/PhD EMP/2020/minute_2020_data.csv')

* So, up to now, we have minute level aggregations on 2020 data and a master session for all 2020 data